# Lab 13 · seaborn, bản đồ số chỗ ở và phản biện cách chọn mốc

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 13**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng bài 13 vẽ choropleth **giá** và phân tích ba biểu đồ lỗi. Trong lab này,
bạn sẽ vẽ bản đồ **số chỗ ở** để bổ sung cho bản đồ giá, so sánh phân phối bằng seaborn
theo một lát cắt mới và lập hồ sơ lỗi cho một hình chọn mốc thời gian có lợi.

*Lab khoảng 50 phút; 30 phút cuối dành cho phần hỗ trợ bài tập lớn ở cuối notebook.*

## Cách làm việc trong bài lab

- Bài tập được chia thành các bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`. Khi tất cả
  `assert` chạy thành công, lời giải đã đáp ứng yêu cầu của bước đó.
- Với phần khởi động và bài có hướng dẫn, bạn nên **tự gõ, không dùng AI**. Các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết sẽ kiểm tra những kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa tiến triển sau 3 phút ở một bước, hãy trao đổi với giảng viên thực hành.

## Mục tiêu

Sau bài lab, bạn sẽ:

1. Dùng seaborn (boxplot và hue) để so sánh phân phối giữa các nhóm trên thang log.
2. Vẽ choropleth **số chỗ ở** và đặt cạnh bản đồ giá để bổ sung thông tin về quy mô nguồn cung.
3. Xử lý quận có mặt trên bản đồ nhưng không có trong bảng dữ liệu (NaN sau phép ghép).
4. Lập hồ sơ lỗi và sửa một hình chọn mốc so sánh có lợi.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

sns.set_theme(style="whitegrid")
BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"
ds = pd.read_csv(f"{BASE}/listings.csv")
rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
len(ds), len(rv)

### Bước 1 · Boxplot và hue: giá theo nhóm chủ nhà (~12 phút)

Biểu đồ so sánh giá của chủ nhà chuyên nghiệp (có từ 5 chỗ ở) với chủ nhà cá nhân,
đồng thời tách theo loại phòng.

In [ ]:
ds["kieu_host"] = (ds["calculated_host_listings_count"] >= 5).map(
    {True: "chuyên nghiệp", False: "cá nhân"})
co_gia = ds.dropna(subset=["price"])

# TODO: sns.boxplot — data=co_gia, x="price", y="room_type", hue="kieu_host",
#       rồi đặt thang log cho trục x (ax.set_xscale("log")) và tiêu đề nêu thông điệp
fig, ax = plt.subplots(figsize=(8.5, 4))
...

# --- Ô kiểm tra ---
assert ax.get_xscale() == "log", "Giá lệch phải — boxplot phải xem trên thang log"
assert len(ax.get_title()) >= 15
assert ax.get_legend() is not None, "hue phải sinh chú giải"
print("Boxplot 3 chiều đạt chuẩn.")

Ở mọi loại phòng, phân phối của nhóm chủ nhà chuyên nghiệp dịch nhẹ về bên phải, phù hợp
với kết quả ở lab 5. Hai nhóm vẫn chồng lấn nhiều, nên mức chênh lệch không tách chúng
thành hai phân khúc riêng biệt.

### Bước 2 · Cặp bản đồ giá và số chỗ ở (~20 phút)

Bản đồ giá dễ làm các quận có diện tích lớn thu hút quá nhiều sự chú ý. Đặt thêm bản đồ
**số chỗ ở** bên cạnh sẽ cho biết quy mô nguồn cung của từng quận.

In [ ]:
geo = gpd.read_file(f"{BASE}/neighbourhoods.geojson")

# TODO: tính số chỗ ở mỗi quận (groupby size, reset_index, đặt tên cột "n"),
#       merge vào geo (how="left"), đếm số quận NaN
kpi_n = ...
ban_do = ...
so_quan_nan = ...

# --- Ô kiểm tra ---
assert len(geo) == 32 and len(ban_do) == 32
assert so_quan_nan == 1
print("32 quận trên bản đồ, 31 quận có chỗ ở — 1 quận trắng dữ liệu.")

Một quận có ranh giới nhưng **không có chỗ ở nào trong mốc chụp này**, nên GeoJSON có nhiều
quận hơn bảng dữ liệu. Trong trường hợp này, NaN sau phép ghép được đổi thành **0** và cần
nêu rõ trong chú thích; đây không phải giá trị "chưa biết" như ở một số bài trước.

In [ ]:
# TODO: điền 0 cho n, rồi vẽ choropleth số chỗ ở:
#       ban_do.plot(column="n", cmap="Blues", legend=True, edgecolor="#999", ax=ax)
#       + tiêu đề nêu thông điệp; tắt trục (ax.set_axis_off())
ban_do["n"] = ...
fig, ax = plt.subplots(figsize=(7, 7))
...

# --- Ô kiểm tra ---
assert ban_do["n"].isna().sum() == 0 and ban_do["n"].max() == 7182
assert not ax.axison, "Bản đồ nên tắt khung trục toạ độ"
print("Bản đồ số chỗ ở cho thấy nguồn cung tập trung ở các quận trung tâm có diện tích nhỏ.")

Đặt bản đồ này cạnh bản đồ giá của notebook bài giảng sẽ cho thấy Lo Barnechea có giá cao
nhưng ít chỗ ở, còn quận Santiago có giá thấp hơn nhưng chiếm 39% số chỗ ở. Chỉ bản đồ giá
không thể hiện được quy mô nguồn cung.

### Bước 3 · Hồ sơ lỗi cho hình chọn mốc có lợi (~12 phút)

Ô dưới dùng số liệu đúng nhưng chọn mốc bắt đầu làm kết luận bị phóng đại.

In [ ]:
nam = rv.set_index("date").resample("YE").size()
tu_2020 = nam.loc["2020":"2025"]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(tu_2020.index.year, tu_2020.values, marker="o", color="#E62727", lw=2)
ax.set_title(f"BÙNG NỔ x{nam.loc['2025-12-31'] / nam.loc['2020-12-31']:.0f}: đánh giá tăng "
             f"{nam.loc['2025-12-31'] / nam.loc['2020-12-31']:.0f} lần kể từ 2020!")
plt.show()

In [ ]:
# Hồ sơ lỗi (điền 2 dòng trả lời vào chuỗi rồi chạy):
ho_so_loi = """
Lỗi 1 (mốc so sánh): ...
Lỗi 2 (lời so với số): ...
"""

# TODO: vẽ BẢN SỬA — cùng dữ liệu nhưng từ 2016, bỏ năm cụt 2026,
#       đánh dấu giai đoạn COVID; đặt tiêu đề phù hợp với dữ liệu
du_lieu_sua = nam.loc["2016":"2025"]
fig, ax = plt.subplots(figsize=(8, 3.2))
...

# --- Ô kiểm tra ---
assert len(ho_so_loi.strip().splitlines()) >= 2 and "..." not in ho_so_loi
assert du_lieu_sua.index.year.min() == 2016
assert any(len(l.get_xdata()) >= 8 for l in ax.lines), "bản sửa phải trải nhiều năm (từ 2016) — dài hơn hình gốc 2020–2025"
print("Bản sửa cho thấy năm 2020 là một đáy bất thường; chọn năm này làm mốc sẽ phóng đại mức tăng.")

## Bài tự làm ✅ mở

**Tự làm 1 · Biểu đồ tương tác.** Dùng `plotly.express.scatter` vẽ kinh độ, vĩ độ của 5.000 chỗ ở
với `hover_name="name"`, `hover_data=["neighbourhood", "price"]`. Kiểm tra năm điểm
có giá cao bất thường và viết hai dòng về thông tin thao tác rê chuột cung cấp, cùng lý do báo cáo
PDF vẫn cần hình tĩnh.

**Tự làm 2 · Cặp bản đồ cho thành phố của nhóm.** Vẽ cặp choropleth gồm giá trung vị và số chỗ ở
cho thành phố bài tập lớn của nhóm. Mỗi mốc chụp Inside Airbnb đều có
`neighbourhoods.geojson`. Hãy xử lý quận không có chỗ ở và ghi cỡ mẫu `n` trong chú thích.

In [ ]:
# Viết bài tự làm của bạn ở đây

---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm theo nhóm)

Trọng tâm: **kiểm tra chất lượng hình và tiến độ hợp phần LLM.**

1. ☐ Dùng **năm câu hỏi phản biện** ở slide 13 để kiểm tra từng hình của nhóm:
   trục, đơn vị và thang đo, cỡ mẫu n, dạng biểu đồ, lời diễn giải. Ghi lại hình chưa đạt.
2. ☐ Nếu có hình bản đồ: đã kèm bản đồ số chỗ ở hoặc chú thích cỡ mẫu n chưa?
3. ☐ Hợp phần LLM: đã gán tay ít nhất 100 nhãn, mỗi đánh giá do hai người gán;
   đã có kết quả độ chính xác đầu tiên của LLM và baseline.
4. ☐ So sánh giữa các mốc chụp bằng cùng thang đo; quy về chỉ số
   khi hai thành phố dùng đơn vị tiền tệ khác nhau, như ví dụ Hình C ở slide bài 13.
5. ☐ Kế hoạch hai tuần cuối nêu rõ người phụ trách từng mục và hạn hoàn thành bản nháp
   (khuyến nghị: trước bài 14 ba ngày).

> Nhóm hoàn thành sớm: đổi hình cho nhóm bên cạnh và phản biện chéo bằng năm câu hỏi.

## Tóm tắt bài lab

| Bạn đã làm | Dùng cho |
|---|---|
| Boxplot và hue trên thang log | so sánh phân phối nhiều chiều trong báo cáo |
| Cặp bản đồ giá và số chỗ ở; đổi NaN thành 0 có chủ đích | bổ sung quy mô nguồn cung cho bản đồ giá |
| Hồ sơ lỗi và bản sửa hình chọn mốc có lợi | phản biện hình do AI sinh (bài 13–14) |

Bài giảng tiếp theo: **kể chuyện bằng dữ liệu và thẩm định báo cáo do AI tạo**.